In [ ]:
import os
import shutil

DIFFSHIELD_DIR = '/kaggle/working/DiffShield'
EOT_PATH = os.path.join(DIFFSHIELD_DIR, 'src/eot.py')

# Robust idempotent clone: Wipes corrupted phantom directories if download was interrupted
if not os.path.exists(EOT_PATH):
    if os.path.exists(DIFFSHIELD_DIR):
        print(f"Corrupted {DIFFSHIELD_DIR} detected. Wiping and re-cloning...")
        shutil.rmtree(DIFFSHIELD_DIR)
    
    !git clone https://github.com/Project-DiffShield/DiffShield.git {DIFFSHIELD_DIR}
else:
    print(f"Repository already cloned and verified.")

import sys
if DIFFSHIELD_DIR not in sys.path:
    sys.path.append(DIFFSHIELD_DIR)

!pip install -q kornia diffusers transformers optuna lpips accelerate scikit-learn kneed matplotlib pandas scikit-image
print("Environment dependencies initialized.")

# Mathematical Patch: Forces Expectation over Transformation to apply identically across paired images
with open(EOT_PATH, 'r') as f:
    eot_src = f.read()

if 'same_on_batch=True' in eot_src:
    print("src/eot.py already patched — skipping.")
else:
    eot_src_patched = eot_src.replace(
        "self.jpeg = K.RandomJPEG(jpeg_quality=jpeg_quality, p=0.5)",
        "self.jpeg = K.RandomJPEG(jpeg_quality=jpeg_quality, p=0.5, same_on_batch=True)"
    ).replace(
        "self.crop = K.RandomResizedCrop(\n            size=(image_size, image_size), scale=crop_scale, p=0.5\n        )",
        "self.crop = K.RandomResizedCrop(\n            size=(image_size, image_size), scale=crop_scale, p=0.5, same_on_batch=True\n        )"
    )
    assert eot_src_patched != eot_src, "EoT patch did not apply. Check source formatting."
    with open(EOT_PATH, 'w') as f:
        f.write(eot_src_patched)
    print("Patched src/eot.py (same_on_batch=True for paired transforms).")

# GPU Memory Patch: Injects global loss_fn to prevent VRAM overflow
optimizer_path = os.path.join(DIFFSHIELD_DIR, 'src/optimizer.py')
with open(optimizer_path, 'r') as f:
    opt_src = f.read()

if 'loss_fn=None' in opt_src:
    print("src/optimizer.py already patched — skipping.")
else:
    opt_src_patched = opt_src.replace(
        "def __init__(self, epsilon=8/255, alpha=2/255, iters=40, device='cpu'):\n"
        "        self.epsilon = epsilon\n"
        "        self.alpha = alpha\n"
        "        self.iters = iters\n"
        "        self.device = device\n"
        "        \n"
        "        self.eot = EoTLayer().to(self.device)\n"
        "        self.loss_fn = DiffShieldLoss(device=self.device)",
        "def __init__(self, epsilon=8/255, alpha=2/255, iters=40, device='cpu', loss_fn=None):\n"
        "        self.epsilon = epsilon\n"
        "        self.alpha = alpha\n"
        "        self.iters = iters\n"
        "        self.device = device\n"
        "        \n"
        "        self.eot = EoTLayer().to(self.device)\n"
        "        self.loss_fn = loss_fn if loss_fn is not None else DiffShieldLoss(device=self.device)"
    )
    assert opt_src_patched != opt_src, "Optimizer patch did not apply. Check source formatting."
    with open(optimizer_path, 'w') as f:
        f.write(opt_src_patched)
    print("Patched src/optimizer.py (loss_fn injection support).")

# Structural-Loss Fix: src/losses.py's compute_structure_loss() was implemented as
# a THIRD DISRUPTION channel (Canny-edge-magnitude MSE, added with a +gamma sign
# inside an ascent-maximized total_loss) rather than a fidelity-preservation term.
# That is why raising gamma made SSIM worse instead of better, and why every
# Optuna trial and every boundary sweep in this notebook failed the SSIM>=0.95
# constraint. This patch replaces it with a differentiable 11x11 Gaussian-windowed
# SSIM term (1 - SSIM) and flips forward() to SUBTRACT it, so PGD ascent is now
# penalized for damaging structure instead of rewarded for it.
losses_path = os.path.join(DIFFSHIELD_DIR, 'src/losses.py')
with open(losses_path, 'r') as f:
    losses_src = f.read()

if '_gaussian_window' in losses_src:
    print("src/losses.py already patched — skipping.")
else:
    old_structure_method = (
        "    def compute_structure_loss(self, clean_image, poisoned_image):\n"
        "        \"\"\"\n"
        "        L_str = ||Canny_mag(X_aug) - Canny_mag(X)||^2_2\n"
        "        \n"
        "        Uses the gradient MAGNITUDE from Kornia's Canny detector (not the binary\n"
        "        thresholded edges) because:\n"
        "        - Gradient magnitude is fully differentiable (smooth function of pixel values)\n"
        "        - Binary thresholding involves non-differentiable operations (NMS + hysteresis)\n"
        "        - The magnitude captures edge strength continuously, giving useful gradients\n"
        "        \n"
        "        Maximizing this MSE injects fake micro-edges that disrupt ControlNet's\n"
        "        geometric constraints.\n"
        "        \"\"\"\n"
        "        # Kornia Canny expects [0, 1] range\n"
        "        clean_01 = (clean_image + 1.0) / 2.0\n"
        "        poison_01 = (poisoned_image + 1.0) / 2.0\n"
        "        \n"
        "        # canny() returns (magnitude, edges) — we use magnitude for differentiability\n"
        "        clean_magnitude, _ = KF.canny(clean_01)\n"
        "        poison_magnitude, _ = KF.canny(poison_01)\n"
        "        \n"
        "        return self.mse_loss(poison_magnitude, clean_magnitude)"
    )

    new_structure_method = (
        "    def _gaussian_window(self, window_size=11, sigma=1.5, channels=3, device=\"cpu\", dtype=torch.float32):\n"
        "        coords = torch.arange(window_size, dtype=dtype, device=device) - window_size // 2\n"
        "        g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))\n"
        "        g = (g / g.sum()).unsqueeze(1)\n"
        "        window_2d = g @ g.t()\n"
        "        window = window_2d.expand(channels, 1, window_size, window_size).contiguous()\n"
        "        return window\n"
        "\n"
        "    def compute_structure_loss(self, clean_image, poisoned_image):\n"
        "        \"\"\"\n"
        "        L_str = 1 - SSIM_windowed(X, X_aug)   (11x11 Gaussian window, sigma=1.5)\n"
        "\n"
        "        FIDELITY-PRESERVATION term (unlike L_vis / L_sem, which are disruption\n"
        "        terms). We want PGD to MINIMIZE this quantity, i.e. keep windowed SSIM\n"
        "        high, so it must be SUBTRACTED in forward(), not added.\n"
        "        \"\"\"\n"
        "        clean_01 = (clean_image + 1.0) / 2.0\n"
        "        poison_01 = (poisoned_image + 1.0) / 2.0\n"
        "\n"
        "        C = clean_01.shape[1]\n"
        "        window = self._gaussian_window(11, 1.5, C, clean_01.device, clean_01.dtype)\n"
        "        pad = 11 // 2\n"
        "\n"
        "        mu_x = F.conv2d(clean_01, window, padding=pad, groups=C)\n"
        "        mu_y = F.conv2d(poison_01, window, padding=pad, groups=C)\n"
        "        mu_x_sq, mu_y_sq, mu_xy = mu_x ** 2, mu_y ** 2, mu_x * mu_y\n"
        "\n"
        "        sigma_x_sq = F.conv2d(clean_01 * clean_01, window, padding=pad, groups=C) - mu_x_sq\n"
        "        sigma_y_sq = F.conv2d(poison_01 * poison_01, window, padding=pad, groups=C) - mu_y_sq\n"
        "        sigma_xy = F.conv2d(clean_01 * poison_01, window, padding=pad, groups=C) - mu_xy\n"
        "\n"
        "        C1, C2 = 0.01 ** 2, 0.03 ** 2\n"
        "        ssim_map = ((2 * mu_xy + C1) * (2 * sigma_xy + C2)) / \\\n"
        "                   ((mu_x_sq + mu_y_sq + C1) * (sigma_x_sq + sigma_y_sq + C2))\n"
        "\n"
        "        return 1.0 - ssim_map.mean()"
    )

    old_forward = (
        "    def forward(self, clean_image, poisoned_image, target_concept_embedding, alpha=1.0, beta=1.0, gamma=1.0):\n"
        "        \"\"\"\n"
        "        Tri-Modal Loss: max_\u03b4 (\u03b1\u00b7L_vis - \u03b2\u00b7L_sem + \u03b3\u00b7L_str)\n"
        "        \n"
        "        Sign convention explanation:\n"
        "        - L_vis (MSE in VAE latent): We MAXIMIZE this \u2192 destroys spatial blueprint\n"
        "        - L_sem (1 - cos_sim):       We MINIMIZE this \u2192 pushes cos_sim toward 1.0\n"
        "                                      (image embedding \u2192 \"a potted plant\")\n"
        "        - L_str (MSE of edge magnitudes): We MAXIMIZE this \u2192 injects fake edges\n"
        "        \n"
        "        Since PGD performs gradient ASCENT on total_loss:\n"
        "        - +\u03b1\u00b7L_vis: ascending maximizes latent divergence \u2713\n"
        "        - -\u03b2\u00b7L_sem: ascending minimizes (1-cos), i.e. maximizes cos_sim \u2713\n"
        "        - +\u03b3\u00b7L_str: ascending maximizes edge divergence \u2713\n"
        "        \"\"\"\n"
        "        l_vis = self.compute_visual_loss(clean_image, poisoned_image)\n"
        "        l_sem = self.compute_semantic_loss(poisoned_image, target_concept_embedding)\n"
        "        l_str = self.compute_structure_loss(clean_image, poisoned_image)\n"
        "        \n"
        "        # Objective to MAXIMIZE via PGD gradient ascent\n"
        "        total_loss = (alpha * l_vis) - (beta * l_sem) + (gamma * l_str)\n"
        "        return total_loss, l_vis, l_sem, l_str"
    )

    new_forward = (
        "    def forward(self, clean_image, poisoned_image, target_concept_embedding, alpha=1.0, beta=1.0, gamma=1.0):\n"
        "        \"\"\"\n"
        "        Tri-Modal Loss: max_delta (alpha*L_vis - beta*L_sem - gamma*L_str)\n"
        "\n"
        "        Sign convention explanation:\n"
        "        - L_vis (MSE in VAE latent):  We MAXIMIZE this -> destroys spatial blueprint\n"
        "        - L_sem (1 - cos_sim):        We MINIMIZE this -> pushes cos_sim toward 1.0\n"
        "                                       (image embedding -> \"a potted plant\")\n"
        "        - L_str (1 - windowed SSIM):  We MINIMIZE this -> PRESERVES structural fidelity\n"
        "\n"
        "        Since PGD performs gradient ASCENT on total_loss:\n"
        "        - +alpha*L_vis: ascending maximizes latent divergence (attack)\n"
        "        - -beta*L_sem:  ascending minimizes (1-cos), i.e. maximizes cos_sim (attack)\n"
        "        - -gamma*L_str: ascending minimizes (1-SSIM), i.e. keeps SSIM high (preservation)\n"
        "        \"\"\"\n"
        "        l_vis = self.compute_visual_loss(clean_image, poisoned_image)\n"
        "        l_sem = self.compute_semantic_loss(poisoned_image, target_concept_embedding)\n"
        "        l_str = self.compute_structure_loss(clean_image, poisoned_image)\n"
        "\n"
        "        # Objective to MAXIMIZE via PGD gradient ascent\n"
        "        total_loss = (alpha * l_vis) - (beta * l_sem) - (gamma * l_str)\n"
        "        return total_loss, l_vis, l_sem, l_str"
    )

    assert old_structure_method in losses_src, "compute_structure_loss source not found — repo version may have changed."
    assert old_forward in losses_src, "forward() source not found — repo version may have changed."

    losses_src_patched = losses_src.replace(old_structure_method, new_structure_method).replace(old_forward, new_forward)
    losses_src_patched = losses_src_patched.replace(
        "import kornia.filters as KF",
        "import kornia.filters as KF\nimport torch.nn.functional as F"
    )
    assert losses_src_patched != losses_src, "Structural-loss patch did not apply. Check source formatting."
    with open(losses_path, 'w') as f:
        f.write(losses_src_patched)
    print("Patched src/losses.py (structural loss now preserves SSIM instead of attacking it).")


In [ ]:
import torch
import numpy as np
import math
import os
import shutil
import zipfile
import json
import pandas as pd
import matplotlib.pyplot as plt
import torchvision.transforms as T
from PIL import Image
import lpips
from torchvision.utils import save_image, make_grid
from skimage.metrics import structural_similarity as ssim_func

from src.losses import DiffShieldLoss
from src.optimizer import PGDOptimizer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Executing on device: {device}")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

ARTIFACTS_DIR = '/kaggle/working/artifacts'
METRICS_DIR = '/kaggle/working/metrics'
os.makedirs(ARTIFACTS_DIR, exist_ok=True)
os.makedirs(METRICS_DIR, exist_ok=True)

loss_fn = DiffShieldLoss(device=device)
lpips_vgg = lpips.LPIPS(net='vgg').to(device)
target_concept_embedding = loss_fn.encode_target_text(["a potted plant"])

global_optimizer = PGDOptimizer(epsilon=8/255, alpha=1/255, iters=40, device=device, loss_fn=loss_fn)

def compute_image_quality_metrics(clean_tensor, immunized_tensor):
    clean_np = ((clean_tensor.squeeze(0).permute(1, 2, 0).cpu().numpy() + 1.0) * 127.5).astype(np.uint8)
    immunized_np = ((immunized_tensor.squeeze(0).permute(1, 2, 0).cpu().numpy() + 1.0) * 127.5).astype(np.uint8)
    
    mse = np.mean((clean_np.astype(np.float64) - immunized_np.astype(np.float64)) ** 2)
    psnr = 20 * math.log10(255.0 / math.sqrt(mse)) if mse > 0 else float('inf')
    
    ssim = ssim_func(
        clean_np, immunized_np,
        channel_axis=-1, data_range=255,
        gaussian_weights=True, sigma=1.5, use_sample_covariance=False
    )
    
    with torch.no_grad():
        lpips_score = lpips_vgg(clean_tensor, immunized_tensor).item()
        
    linf = (immunized_tensor - clean_tensor).abs().max().item()
    return {"MSE": mse, "PSNR": psnr, "SSIM": ssim, "LPIPS": lpips_score, "Linf": linf}

print("Backbones, global optimizer, and standardized metric computation functions initialized.")

In [ ]:
dataset_base = None
img_dir = None
attr_file_path = None

for root, dirs, files in os.walk('/kaggle/input'):
    if 'CelebA-HQ-img' in dirs and 'CelebAMask-HQ-attribute-anno.txt' in files:
        dataset_base = root
        img_dir = os.path.join(root, 'CelebA-HQ-img')
        attr_file_path = os.path.join(root, 'CelebAMask-HQ-attribute-anno.txt')
        break

if not img_dir or not attr_file_path:
    raise FileNotFoundError("Could not locate CelebA-HQ-img or CelebAMask-HQ-attribute-anno.txt.")

with open(attr_file_path, 'r') as f:
    lines = [line.strip() for line in f.readlines() if line.strip()]

num_images = int(lines[0])
attr_names = lines[1].split()
data = []
img_filenames = []

for line in lines[2:]:
    parts = line.split()
    img_filenames.append(parts[0])
    data.append([1 if int(x) == 1 else 0 for x in parts[1:]])

attr_matrix = np.array(data, dtype=np.float32)
print(f"Loaded {attr_matrix.shape[0]} images across {attr_matrix.shape[1]} binary attributes.")

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances_argmin_min
from kneed import KneeLocator

wcss = []
k_range = list(range(1, 21))

print("Computing WCSS across cluster ranges (1 to 20)...")
for k in k_range:
    km = KMeans(n_clusters=k, init='k-means++', random_state=42, n_init=10)
    km.fit(attr_matrix)
    wcss.append(km.inertia_)

wcss_df = pd.DataFrame({"K": k_range, "WCSS": wcss})
wcss_df.to_csv(os.path.join(METRICS_DIR, 'wcss_elbow_values.csv'), index=False)

kl = KneeLocator(k_range, wcss, curve="convex", direction="decreasing")
k_calib = int(kl.elbow) if kl.elbow is not None else 4
print(f"Mathematical Elbow Detected at K = {k_calib}")

plt.figure(figsize=(8, 5))
plt.plot(k_range, wcss, marker='o', color='#A4123F', linewidth=2, markersize=6)
plt.axvline(x=k_calib, color='navy', linestyle='--', label=f'Optimal K ({k_calib})')
plt.title('Elbow Method: Attribute Variance Clustering', fontsize=12, fontweight='bold')
plt.xlabel('Number of Clusters (K)', fontsize=11)
plt.ylabel('Within-Cluster Sum of Squares (WCSS)', fontsize=11)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend()
elbow_plot_path = os.path.join(ARTIFACTS_DIR, 'elbow_method_wcss_curve.png')
plt.savefig(elbow_plot_path, dpi=300, bbox_inches='tight')
plt.close()

kmeans_calib = KMeans(n_clusters=k_calib, init='k-means++', random_state=42, n_init=10)
kmeans_calib.fit(attr_matrix)
centroid_indices_calib, _ = pairwise_distances_argmin_min(kmeans_calib.cluster_centers_, attr_matrix)
calib_filenames = [img_filenames[idx] for idx in centroid_indices_calib]

calib_manifest_df = pd.DataFrame({
    "Calibration_Cluster_ID": list(range(1, k_calib + 1)),
    "Original_Index": centroid_indices_calib,
    "Filename": calib_filenames
})
calib_manifest_df.to_csv(os.path.join(METRICS_DIR, 'calibration_subset_manifest.csv'), index=False)

calib_dir = '/kaggle/working/calibration_subset'
os.makedirs(calib_dir, exist_ok=True)

for fname in calib_filenames:
    src_path = os.path.join(img_dir, fname)
    if os.path.exists(src_path):
        img = Image.open(src_path).convert('RGB')
        img_resized = img.resize((512, 512), Image.Resampling.BICUBIC)
        img_resized.save(os.path.join(calib_dir, fname))

print(f"Calibration Manifest saved. Stored {len(os.listdir(calib_dir))} calibration centroids in {calib_dir}")

In [ ]:
calib_set = set(centroid_indices_calib)
eval_indices_available = [i for i in range(len(img_filenames)) if i not in calib_set]

eval_attr_matrix = attr_matrix[eval_indices_available]
eval_filenames_available = [img_filenames[i] for i in eval_indices_available]

K_EVAL = 70
print(f"Clustering {eval_attr_matrix.shape[0]} disjoint images into {K_EVAL} evaluation centroids...")
kmeans_eval = KMeans(n_clusters=K_EVAL, init='k-means++', random_state=42, n_init=10)
kmeans_eval.fit(eval_attr_matrix)

centroid_indices_eval, _ = pairwise_distances_argmin_min(kmeans_eval.cluster_centers_, eval_attr_matrix)
eval_final_filenames = [eval_filenames_available[idx] for idx in centroid_indices_eval]

eval_manifest_df = pd.DataFrame({
    "Eval_Image_ID": [f"face_{i+1:03d}" for i in range(K_EVAL)],
    "Original_Index": [eval_indices_available[idx] for idx in centroid_indices_eval],
    "Filename": eval_final_filenames
})
eval_manifest_df.to_csv(os.path.join(METRICS_DIR, 'evaluation_70_manifest.csv'), index=False)

eval_dir = '/kaggle/working/diverse_70_images'
os.makedirs(eval_dir, exist_ok=True)

for fname in eval_final_filenames:
    src_path = os.path.join(img_dir, fname)
    if os.path.exists(src_path):
        img = Image.open(src_path).convert('RGB')
        img_resized = img.resize((512, 512), Image.Resampling.BICUBIC)
        img_resized.save(os.path.join(eval_dir, fname))

print(f"Data isolation complete. Stored 70 evaluation centroids in {eval_dir}")

In [ ]:
from torch.utils.data import DataLoader
from src.data import get_dataloader

calib_loader = get_dataloader(root_dir=calib_dir, batch_size=int(k_calib), image_size=512)
calib_batch, _ = next(iter(calib_loader))
calib_batch = calib_batch.to(device)

epsilon = 8 / 255
delta = torch.zeros_like(calib_batch).to(device)
delta.uniform_(-epsilon, epsilon)
poisoned_batch = torch.clamp(calib_batch + delta, -1.0, 1.0)

raw_vis = loss_fn.compute_visual_loss(calib_batch, poisoned_batch).item()
raw_sem = loss_fn.compute_semantic_loss(poisoned_batch, target_concept_embedding).item()
raw_str = loss_fn.compute_structure_loss(calib_batch, poisoned_batch).item()

alpha_base = 1.0 / max(raw_vis, 1e-4)
beta_base  = 1.0 / max(raw_sem, 1e-4)
gamma_base = 1.0 / max(raw_str, 1e-4)

base_config = {
    "raw_losses": {"visual": raw_vis, "semantic": raw_sem, "structural": raw_str},
    "base_multipliers": {"alpha_base": alpha_base, "beta_base": beta_base, "gamma_base": gamma_base}
}
with open(os.path.join(METRICS_DIR, 'base_multipliers.json'), 'w') as f:
    json.dump(base_config, f, indent=4)

print(f"Base multipliers calculated and saved: Alpha={alpha_base:.4f}, Beta={beta_base:.4f}, Gamma={gamma_base:.4f}")

In [ ]:
multipliers = [0.1, 0.25, 0.5, 1.0, 2.0, 3.0, 5.0]
sweep_logs = []

def sweep_subset_bounds(param_name):
    valid_mults = []
    print(f"\n--- Sweeping Search Boundaries for {param_name} ---")
    
    for mult in multipliers:
        w_a = alpha_base * (mult if param_name == 'alpha' else 1.0)
        w_b = beta_base  * (mult if param_name == 'beta'  else 1.0)
        w_g = gamma_base * (mult if param_name == 'gamma' else 1.0)
        
        batch_passed = True
        min_psnr_batch = float('inf')
        min_ssim_batch = float('inf')
        
        for i in range(calib_batch.size(0)):
            img = calib_batch[i:i+1]
            immunized = global_optimizer.optimize(img, target_concept_embedding, w_alpha=w_a, w_beta=w_b, w_gamma=w_g)
            m = compute_image_quality_metrics(img, immunized)
            min_psnr_batch = min(min_psnr_batch, m['PSNR'])
            min_ssim_batch = min(min_ssim_batch, m['SSIM'])
            
            if m['PSNR'] < 38.0 or m['SSIM'] < 0.95:
                batch_passed = False
                break
                
        status = "PASS" if batch_passed else "FAIL"
        sweep_logs.append({
            "parameter": param_name, "multiplier": mult,
            "min_batch_psnr": min_psnr_batch, "min_batch_ssim": min_ssim_batch,
            "status": status
        })
        print(f"Multiplier {mult:4.2f}x | Min PSNR: {min_psnr_batch:.2f} dB | Min SSIM: {min_ssim_batch:.4f} | Status: {status}")
        if batch_passed:
            valid_mults.append(mult)
            
    min_m = min(valid_mults) if valid_mults else 0.1
    max_m = max(valid_mults) if valid_mults else 1.0
    return min_m, max_m

min_a, max_a = sweep_subset_bounds('alpha')
min_b, max_b = sweep_subset_bounds('beta')
min_g, max_g = sweep_subset_bounds('gamma')

min_alpha, max_alpha = alpha_base * min_a, alpha_base * max_a
min_beta,  max_beta  = beta_base  * min_b, beta_base  * max_b
min_gamma, max_gamma = gamma_base * min_g, gamma_base * max_g

pd.DataFrame(sweep_logs).to_csv(os.path.join(METRICS_DIR, 'boundary_sweeps_log.csv'), index=False)

bounds_dict = {
    "alpha_bounds": [min_alpha, max_alpha],
    "beta_bounds": [min_beta, max_beta],
    "gamma_bounds": [min_gamma, max_gamma]
}
with open(os.path.join(METRICS_DIR, 'optuna_search_boundaries.json'), 'w') as f:
    json.dump(bounds_dict, f, indent=4)

print(f"Search boundaries persisted to {METRICS_DIR}/optuna_search_boundaries.json")

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    w_a = trial.suggest_float('alpha', min_alpha, max_alpha)
    w_b = trial.suggest_float('beta',  min_beta,  max_beta)
    w_g = trial.suggest_float('gamma', min_gamma, max_gamma)
    
    subset_lpips_scores = []
    worst_psnr_deficit = 0.0
    worst_ssim_deficit = 0.0
    
    for i in range(calib_batch.size(0)):
        img = calib_batch[i:i+1]
        immunized = global_optimizer.optimize(img, target_concept_embedding, w_alpha=w_a, w_beta=w_b, w_gamma=w_g)
        m = compute_image_quality_metrics(img, immunized)
        
        worst_psnr_deficit = max(worst_psnr_deficit, max(0.0, 38.0 - m['PSNR']))
        worst_ssim_deficit = max(worst_ssim_deficit, max(0.0, 0.95 - m['SSIM']))
        subset_lpips_scores.append(m['LPIPS'])
    
    # Feasible: reward worst-case LPIPS directly, as before.
    if worst_psnr_deficit == 0.0 and worst_ssim_deficit == 0.0:
        return min(subset_lpips_scores)
    
    # Infeasible: instead of a flat -9999.0 (which gives Optuna's TPE sampler
    # zero signal to distinguish a near-miss from a total miss), return a graded
    # penalty. This still ranks every feasible trial above every infeasible one
    # (LPIPS values are always well above -100), while letting the sampler learn
    # which region of (alpha, beta, gamma) is getting closer to feasible.
    return -100.0 - worst_psnr_deficit - (worst_ssim_deficit * 100.0)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

opt_alpha = study.best_params['alpha']
opt_beta  = study.best_params['beta']
opt_gamma = study.best_params['gamma']

trials_df = study.trials_dataframe()
trials_df.to_csv(os.path.join(METRICS_DIR, 'optuna_all_trials_log.csv'), index=False)

best_params_record = {
    "best_trial_number": study.best_trial.number,
    "best_objective_value": study.best_value,
    "optimal_weights": {
        "alpha": opt_alpha,
        "beta": opt_beta,
        "gamma": opt_gamma
    }
}
with open(os.path.join(METRICS_DIR, 'optimal_hyperparameters.json'), 'w') as f:
    json.dump(best_params_record, f, indent=4)

valid_scores = [t.value for t in study.trials if t.value is not None and t.value > -9000]
plt.figure(figsize=(8, 4))
plt.plot(range(1, len(valid_scores) + 1), valid_scores, marker='s', color='darkgreen', linewidth=2)
plt.title('Optuna Bayesian Optimization: Worst-Case (Min) LPIPS Progression', fontsize=12, fontweight='bold')
plt.xlabel('Valid Trial Count', fontsize=11)
plt.ylabel('Worst-Case LPIPS (Higher = Better)', fontsize=11)
plt.grid(True, linestyle=':', alpha=0.6)
optuna_plot_path = os.path.join(ARTIFACTS_DIR, 'optuna_convergence_history.png')
plt.savefig(optuna_plot_path, dpi=300, bbox_inches='tight')
plt.close()

print(f"\n==================================================")
print("  FINAL OPTIMAL GENERALIZED HYPERPARAMETERS")
print("==================================================")
print(f"  Optimal Alpha (Visual)     = {opt_alpha:.4f}")
print(f"  Optimal Beta  (Semantic)   = {opt_beta:.4f}")
print(f"  Optimal Gamma (Structural) = {opt_gamma:.4f}")
print(f"  Worst-Case Calibration LPIPS = {study.best_value:.4f}")

In [ ]:
eval_loader = get_dataloader(root_dir=eval_dir, batch_size=1, image_size=512)
output_protected_dir = '/kaggle/working/stage1_protected_images'
os.makedirs(output_protected_dir, exist_ok=True)

detailed_metrics = []
sample_visuals = []

print(f"Processing 70 Disjoint Centroids using Optimal Hyperparameters...")
for idx, (clean_img, path) in enumerate(eval_loader):
    clean_img = clean_img.to(device)
    raw_fname = os.path.basename(path[0])
    
    immunized_img = global_optimizer.optimize(
        clean_img, target_concept_embedding,
        w_alpha=opt_alpha, w_beta=opt_beta, w_gamma=opt_gamma
    )
    
    m = compute_image_quality_metrics(clean_img, immunized_img)
    
    protected_norm = (immunized_img.squeeze(0) + 1.0) / 2.0
    clean_norm = (clean_img.squeeze(0) + 1.0) / 2.0
    
    save_filename = f"protected_face_{idx+1:03d}.png"
    save_image(protected_norm, os.path.join(output_protected_dir, save_filename))
    
    detailed_metrics.append({
        "Image_ID": f"face_{idx+1:03d}",
        "Original_Filename": raw_fname,
        "Protected_Filename": save_filename,
        "PSNR_dB": m['PSNR'],
        "SSIM": m['SSIM'],
        "LPIPS": m['LPIPS'],
        "Linf": m['Linf'],
        "MSE": m['MSE'],
        "PSNR_Passed": m['PSNR'] >= 38.0,
        "SSIM_Passed": m['SSIM'] >= 0.95
    })
    
    if idx < 4:
        sample_visuals.extend([clean_norm.cpu(), protected_norm.cpu()])
        
    if (idx + 1) % 10 == 0 or (idx + 1) == 70:
        print(f"[{idx+1:02d}/70] -> PSNR: {m['PSNR']:.2f} dB | SSIM: {m['SSIM']:.4f} | LPIPS: {m['LPIPS']:.4f}")

per_image_df = pd.DataFrame(detailed_metrics)
per_image_df.to_csv(os.path.join(METRICS_DIR, 'stage1_per_image_metrics.csv'), index=False)

grid = make_grid(sample_visuals, nrow=2, padding=10, normalize=False)
grid_np = grid.permute(1, 2, 0).numpy()

plt.figure(figsize=(8, 8))
plt.imshow(grid_np)
plt.axis('off')
plt.title('DiffShield Phase 1: Clean (Left) vs. Protected (Right)', fontsize=12, fontweight='bold')
comp_plot_path = os.path.join(ARTIFACTS_DIR, 'phase1_sample_comparisons.png')
plt.savefig(comp_plot_path, dpi=300, bbox_inches='tight')
plt.close()

In [ ]:
psnr_vals = [r['PSNR_dB'] for r in detailed_metrics]
ssim_vals = [r['SSIM'] for r in detailed_metrics]
lpips_vals = [r['LPIPS'] for r in detailed_metrics]
linf_vals = [r['Linf'] for r in detailed_metrics]

summary_records = [
    {
        "Metric": "PSNR (dB)",
        "Mean": np.mean(psnr_vals), "Std": np.std(psnr_vals),
        "Min": np.min(psnr_vals), "Max": np.max(psnr_vals),
        "Threshold": ">= 38.0",
        "Violations": sum(1 for p in psnr_vals if p < 38.0)
    },
    {
        "Metric": "SSIM (Windowed)",
        "Mean": np.mean(ssim_vals), "Std": np.std(ssim_vals),
        "Min": np.min(ssim_vals), "Max": np.max(ssim_vals),
        "Threshold": ">= 0.95",
        "Violations": sum(1 for s in ssim_vals if s < 0.95)
    },
    {
        "Metric": "LPIPS",
        "Mean": np.mean(lpips_vals), "Std": np.std(lpips_vals),
        "Min": np.min(lpips_vals), "Max": np.max(lpips_vals),
        "Threshold": ">= 0.15 (High=Good)",
        "Violations": sum(1 for l in lpips_vals if l < 0.15)
    },
    {
        "Metric": "Linf",
        "Mean": np.mean(linf_vals), "Std": np.std(linf_vals),
        "Min": np.min(linf_vals), "Max": np.max(linf_vals),
        "Threshold": "<= 0.0314",
        "Violations": sum(1 for li in linf_vals if li > (8/255 + 1e-4))
    }
]

summary_df = pd.DataFrame(summary_records)
summary_df.to_csv(os.path.join(METRICS_DIR, 'stage1_summary_metrics.csv'), index=False)
with open(os.path.join(METRICS_DIR, 'stage1_summary_metrics.json'), 'w') as f:
    json.dump(summary_records, f, indent=4)

print("\n" + "=" * 80)
print(f"  STAGE 1 GENERALIZATION EVALUATION SUMMARY (70 Disjoint Images)")
print("=" * 80)
print(f"{'Metric':<16} | {'Mean ± Std':<18} | {'Min':<10} | {'Max':<10} | {'Threshold':<12} | {'Violations'}")
print("-" * 80)
for r in summary_records:
    print(f"{r['Metric']:<16} | {r['Mean']:6.4f} ± {r['Std']:5.4f}    | {r['Min']:8.4f}   | {r['Max']:8.4f}   | {r['Threshold']:<12} | {r['Violations']}/70")
print("=" * 80)

print("\nCreating downloadable result bundles...")
!zip -r -q /kaggle/working/diffshield_phase1_complete_results.zip /kaggle/working/metrics /kaggle/working/artifacts
!zip -r -q /kaggle/working/stage1_protected_images.zip /kaggle/working/stage1_protected_images
!zip -r -q /kaggle/working/stage1_original_images.zip /kaggle/working/diverse_70_images
print("All artifacts packaged successfully.")